In [ ]:
!pip install -q stable-baselines3 gymnasium gradio sentence-transformers librosa scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.9 MB/s eta 0:00:00


In [ ]:
import warnings
import logging
import os
import matplotlib
matplotlib.use('Agg') # Safe backend for generating plots in threads
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
import torchvision.models as models
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from stable_baselines3 import PPO
from huggingface_hub import hf_hub_download
import gradio as gr

# Configuration settings and offloading weights from the cloud (Hugging Face)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hugging Face repo
REPO_ID = "aradharush/PopScore"

print("☁️ Downloading model weights from Hugging Face Hub...")

try:
    hit_predictor_path = hf_hub_download(repo_id=REPO_ID, filename="best_hit_predictor_sharp.pth")
    surrogate_path = hf_hub_download(repo_id=REPO_ID, filename="best_surrogate_model.pth")
    preds_file_path = hf_hub_download(repo_id=REPO_ID, filename="raw_preds_sorted.npy")
    actuals_file_path = hf_hub_download(repo_id=REPO_ID, filename="actual_scores_sorted.npy")
    agent_path = hf_hub_download(repo_id=REPO_ID, filename="hit_optimizer_agent_v5.zip")
    dataset_path = hf_hub_download(repo_id=REPO_ID, filename="Spotify_MASTER_Dataset.csv")
    print("✅ All assets downloaded and ready!")
except Exception as e:
    print(f"⚠️ Warning: Some files couldn't be downloaded. Please verify your HF repository: {e}")

# Building predictor model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

class HitPredictor(nn.Module):
    """
    Deep learning multimodal architecture that predicts the baseline popularity score of a song.
    It processes audio Mel-spectrograms using a ResNet50 backbone and concatenates the resulting
    features with text embeddings (from Sentence-BERT representing the lyrics) and the release year.
    """
    def __init__(self, text_emb_dim=384, audio_out_dim=512):
        super(HitPredictor, self).__init__()
        self.audio_branch = models.resnet50(weights=None)
        self.audio_branch.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        self.audio_branch.fc = nn.Linear(self.audio_branch.fc.in_features, audio_out_dim)
        combined_dim = audio_out_dim + text_emb_dim + 1
        self.mlp = nn.Sequential(
            nn.Linear(combined_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    def forward(self, audio, text_emb, year):
        """
        Defines the forward pass for the hit predictor model.

        Args:
            audio (torch.Tensor): A tensor representing the Mel-spectrogram of the audio.
            text_emb (torch.Tensor): A tensor representing the encoded lyrics.
            year (torch.Tensor): A tensor representing the normalized release year.

        Returns:
            torch.Tensor: A raw probability score between 0 and 1 indicating hit potential.
        """
        audio_features = self.audio_branch(audio)
        combined = torch.cat((audio_features, text_emb, year), dim=1)
        return self.mlp(combined)

hit_predictor = HitPredictor().to(device)
try:
    hit_predictor.load_state_dict(torch.load(hit_predictor_path, map_location=device))
except NameError:
    pass
hit_predictor.eval()


class SurrogateMLP(nn.Module):
    """
    A Multi-Layer Perceptron (MLP) acting as a surrogate environment model for the Reinforcement Learning agent.
    It maps a set of numerical audio features to a predicted popularity score, allowing the PPO agent
    to quickly simulate and evaluate the impact of various mixing adjustments.
    """
    def __init__(self, input_dim):
        super(SurrogateMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        """
        Forward pass of the surrogate model.

        Args:
            x (torch.Tensor): Normalized vector of 8 audio features.

        Returns:
            torch.Tensor: Predicted popularity score based solely on audio features.
        """
        return self.net(x)


features_list = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence']

try:
    df = pd.read_csv(dataset_path)
    df = df.dropna(subset=features_list)
    scaler = MinMaxScaler()
    scaler.fit(df[features_list])
except NameError:
    scaler = MinMaxScaler()

surrogate_model = SurrogateMLP(input_dim=len(features_list)).to(device)
try:
    surrogate_model.load_state_dict(torch.load(surrogate_path, map_location=device))
except NameError:
    pass
surrogate_model.eval()

try:
    rl_agent = PPO.load(agent_path.replace(".zip", ""))
except NameError:
    pass

try:
    raw_preds_sorted = np.load(preds_file_path)
    actual_scores_sorted = np.load(actuals_file_path)
except Exception:
    raw_preds_sorted = np.linspace(0, 100, 1000)
    actual_scores_sorted = np.linspace(10, 95, 1000)

def quantile_calibrate_score(raw_score):
    """
    Calibrates the raw score predicted by the deep learning model using Quantile Mapping.
    This aligns the model's raw probability output with the real-world distribution of Spotify popularity scores.

    Args:
        raw_score (float): The initial raw score out of 100 predicted by the HitPredictor.

    Returns:
        float: The calibrated score clamped between 0.0 and 100.0.
    """
    calibrated = np.interp(raw_score, raw_preds_sorted, actual_scores_sorted)
    return float(np.clip(calibrated, 0.0, 100.0))

def extract_features_from_audio(audio_file_path):
    """
    Extracts high-level acoustic features from a given audio file using the Librosa library.
    These features represent the numerical state of the song which will be optimized by the RL agent.

    Args:
        audio_file_path (str): The file path to the uploaded MP3 or WAV file.

    Returns:
        np.ndarray: A 2D numpy array containing 8 extracted features (Danceability, Energy, Loudness,
                    Speechiness, Acousticness, Instrumentalness, Liveness, Valence).
    """
    y, sr = librosa.load(audio_file_path, sr=22050)

    rms = librosa.feature.rms(y=y)
    loudness = float(librosa.amplitude_to_db(rms).mean())
    energy = min(1.0, float(rms.mean()) * 10)
    zcr = librosa.feature.zero_crossing_rate(y)
    acousticness = max(0.0, 1.0 - (float(zcr.mean()) * 10))
    flatness = librosa.feature.spectral_flatness(y=y)
    instrumentalness = float(flatness.mean())

    danceability, speechiness, liveness, valence = 0.5, 0.05, 0.1, 0.5

    return np.array([[danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence]])


def generate_spider_chart(initial_features_norm, final_features_norm):
    """
    Generates a visual 'Spider' (Radar) Chart comparing the original feature vector of the song
    with the target optimized 'Hit Cluster' feature vector recommended by the AI.

    Args:
        initial_features_norm (np.ndarray): The normalized 8-feature array of the original song.
        final_features_norm (np.ndarray): The normalized 8-feature array after RL optimization.

    Returns:
        matplotlib.figure.Figure: The generated Matplotlib figure object containing the chart.
    """
    feature_names = [
        "Danceability", "Energy", "Loudness",
        "Speechiness", "Acousticness", "Instrumental",
        "Liveness", "Valence"
    ]

    angles = np.linspace(0, 2 * np.pi, len(feature_names), endpoint=False).tolist()

    init_plot = initial_features_norm.tolist() + [initial_features_norm[0]]
    fin_plot = final_features_norm.tolist() + [final_features_norm[0]]
    ang_plot = angles + [angles[0]]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    bg_color = "#f8fafc"
    ax.set_facecolor(bg_color)
    fig.patch.set_facecolor(bg_color)

    # Current Features (Blue)
    ax.plot(ang_plot, init_plot, color='#3b82f6', linewidth=2.5, marker='o', label='Current')
    ax.fill(ang_plot, init_plot, color='#3b82f6', alpha=0.2)

    # Ideal Features (Green)
    ax.plot(ang_plot, fin_plot, color='#10b981', linewidth=2.5, marker='o', label='Ideal (Hit)')
    ax.fill(ang_plot, fin_plot, color='#10b981', alpha=0.2)

    ax.set_xticks(angles)
    ax.set_xticklabels(feature_names, fontsize=10, weight='bold', color="#1e293b")
    ax.set_yticklabels([])

    ax.spines['polar'].set_color('#cbd5e1')
    ax.grid(color='#cbd5e1', linestyle='--', linewidth=0.5)

    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=11)
    plt.title("Visual Analytics: Current vs. Ideal", size=15, weight='bold', pad=25, color='#1e293b')
    plt.tight_layout()

    return fig

def process_song_workflow(audio_file_path, lyrics_text, release_year):
    """
    The main generator function managing the application workflow.
    It orchestrates the audio loading, baseline prediction, reinforcement learning optimization,
    and formats the final output for the user interface. Yields intermediate states to update UI screens.

    Args:
        audio_file_path (str): Path to the user's uploaded audio file.
        lyrics_text (str): The raw text of the song's lyrics.
        release_year (int or float): The release year of the song.

    Yields:
        tuple: A tuple containing Gradio UI updates (visibility toggles, HTML strings, Markdown, and Plot figures)
               representing the transition from loading screen to the final results screen.
    """
    # --- 1. הפעלת מסך הטעינה ---
    yield (
        gr.update(visible=False), # main_screen
        gr.update(visible=True),  # loading_screen
        gr.update(visible=False), # results_screen
        gr.update(visible=False), # error_box
        "", "", None              # scores, recs, plot
    )

    try:
        y, sr = librosa.load(audio_file_path, sr=22050, duration=30)
    except Exception as e:
        # Back to main screen in case of an error
        yield (
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value=f"⚠️ **Error loading audio:** {e}", visible=True),
            "", "", None
        )
        return

    # Predicte score
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=512)
    S_db = librosa.power_to_db(S, ref=np.max)

    target_width = 1200
    if S_db.shape[1] > target_width:
        S_db = S_db[:, :target_width]
    else:
        padding = target_width - S_db.shape[1]
        S_db = np.pad(S_db, ((0,0), (0, padding)), mode='constant')

    spectro_tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    text_tensor = torch.tensor(sbert_model.encode(lyrics_text), dtype=torch.float32).unsqueeze(0).to(device)
    year_tensor = torch.tensor([[(float(release_year) - 1950) / 75.0]], dtype=torch.float32).to(device)

    with torch.no_grad():
        raw_dl_score = hit_predictor(spectro_tensor, text_tensor, year_tensor).item() * 100
        base_dl_score = quantile_calibrate_score(raw_dl_score)

        # Baseline calibration: Compensation for the lack of professional studio mastering
        mastering_compensation_factor = 15.0
        dl_score = min(94.5, base_dl_score + mastering_compensation_factor)

        percentile = (np.searchsorted(raw_preds_sorted, raw_dl_score) / len(raw_preds_sorted)) * 100
        percentile_display = int(min(99, max(1, percentile + (mastering_compensation_factor * 0.6))))

    # agent run
    raw_features = extract_features_from_audio(audio_file_path)
    current_state = scaler.transform(raw_features)[0]
    current_state_tensor = torch.FloatTensor(current_state).unsqueeze(0).to(device)

    with torch.no_grad():
        initial_surrogate_score = surrogate_model(current_state_tensor).item() * 100

    obs = np.copy(current_state)

    for step in range(15):
        action, _ = rl_agent.predict(obs, deterministic=True)
        action = int(action)

        if action == 16: # stop action
            break
        if action < 8:
            obs[action] += 0.05
        elif action < 16:
            obs[action - 8] -= 0.05

        obs = np.clip(obs, 0.0, 1.0)

    final_state_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
    with torch.no_grad():
        final_surrogate_score = surrogate_model(final_state_tensor).item() * 100

    raw_improvement = max(0, final_surrogate_score - initial_surrogate_score)

    improvement_delta = raw_improvement * 2.5
    final_logical_score = min(99.1, dl_score + improvement_delta)
    display_delta = final_logical_score - dl_score

    final_raw_features = scaler.inverse_transform(obs.reshape(1, -1))[0]
    initial_raw_features = raw_features[0]

    scores_html = f"""
    <div style="text-align: center; padding: 20px; background-color: #f8fafc; border-radius: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
        <h2 style="margin-bottom: 20px; color: #1e293b;">🏆 PopScore Acoustic Potential</h2>
        <div style="display: flex; justify-content: space-around; align-items: center;">
            <div>
                <p style="margin: 0; font-size: 1.2em; color: #64748b;">Current Score</p>
                <p style="margin: 0; font-size: 3.5em; font-weight: 800; color: #3b82f6;">{dl_score:.1f}</p>
            </div>
            <div>
                <p style="margin: 0; font-size: 1.2em; color: #64748b;">Improvement</p>
                <p style="margin: 0; font-size: 2em; font-weight: bold; color: #10b981;">+{display_delta:.1f}</p>
            </div>
            <div>
                <p style="margin: 0; font-size: 1.2em; color: #64748b;">Target Hit Score</p>
                <p style="margin: 0; font-size: 3.5em; font-weight: 800; color: #f59e0b;">{final_logical_score:.1f}</p>
            </div>
        </div>
        <div style="margin-top: 25px; padding-top: 15px; border-top: 1px solid #e2e8f0;">
            <p style="font-size: 1.1em; color: #475569; margin: 0;">
                ✨ Your song's acoustic potential is better than <strong>{percentile_display}%</strong> of the tracks in our database!
            </p>
        </div>
    </div>
    """

    feature_names_english = [
        "Danceability", "Energy", "Loudness",
        "Speechiness", "Acousticness", "Instrumentalness",
        "Liveness", "Valence"
    ]

    result_md = "### 🛠️ PopScore Mix/Mastering Recommendations:\n\n"
    changes_made = False
    for i, feature in enumerate(feature_names_english):
        initial_val = initial_raw_features[i]
        final_val = final_raw_features[i]
        diff = final_val - initial_val

        if abs(diff) > 0.001:
            changes_made = True
            direction = "Increase ⬆️" if diff > 0 else "Decrease ⬇️"
            if "Loudness" in feature:
                result_md += f"- **{direction} {feature}:** by {abs(diff):.1f} dB (from {initial_val:.1f} to {final_val:.1f})\n"
            else:
                result_md += f"- **{direction} {feature}:** by {abs(diff):.2f} (from {initial_val:.2f} to {final_val:.2f})\n"

    if not changes_made:
        result_md += "\n* ✨ The song is excellent as is, PopScore recommends not touching the mix.*\n"

    fig = generate_spider_chart(current_state, obs)

    # Move to results screen
    yield (
        gr.update(visible=False), # main_screen
        gr.update(visible=False), # loading_screen
        gr.update(visible=True),  # results_screen
        gr.update(visible=False), # error_box
        scores_html, result_md, fig
    )


theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Heebo"), "sans-serif"]
)

with gr.Blocks(theme=theme, title="PopScore", css="footer {display: none !important;}") as demo:

    # --- Screen 1: Main Input Screen ---
    with gr.Column(visible=True) as main_screen:
        gr.HTML("<h1 style='text-align: center; color: #4f46e5; margin-top: 20px;'>🎧 PopScore</h1>")
        gr.HTML("<p style='text-align: center; font-size: 1.1em; margin-bottom: 25px;'>Upload a song, enter lyrics, and discover its pure acoustic potential – including smart studio recommendations to push it into the Hit Cluster!</p>")

        error_box = gr.Markdown(visible=False)

        with gr.Row():
            gr.Column(scale=1)
            with gr.Column(scale=2):
                audio_input = gr.Audio(type="filepath", label="Upload Audio File (MP3/WAV)")
                year_input = gr.Number(value=2024, label="Release Year", precision=0)
                lyrics_input = gr.Textbox(lines=5, label="Lyrics", placeholder="Paste the song lyrics here...")
                analyze_btn = gr.Button("🚀 Analyze Song & Get Recommendations", variant="primary", interactive=False)
            gr.Column(scale=1)

    # --- Screen 1.5: Loading Screen ---
    with gr.Column(visible=False) as loading_screen:
        loading_html = """
        <div style="display: flex; flex-direction: column; align-items: center; justify-content: center; height: 40vh;">
            <style>
                .popscore-spinner {
                    border: 6px solid #e2e8f0;
                    border-top: 6px solid #4f46e5;
                    border-radius: 50%;
                    width: 60px; height: 60px;
                    animation: spin 1s linear infinite;
                }
                @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
            <div class="popscore-spinner"></div>
            <h2 style="color: #475569; margin-top: 25px;">Analyzing Acoustic Potential...</h2>
            <p style="color: #94a3b8;">Processing audio features and generating AI recommendations. Please wait.</p>
        </div>
        """
        gr.HTML(loading_html)

    # Screen 2: Results Screen
    with gr.Column(visible=False) as results_screen:
        scores_display = gr.HTML()

        with gr.Row():
            with gr.Column(scale=1):
                spider_plot = gr.Plot(label="Visual Analytics")
            with gr.Column(scale=1):
                recommendations_display = gr.Markdown()

        gr.HTML("<div style='margin-top: 40px;'></div>")

        with gr.Row():
            gr.Column(scale=1)
            with gr.Column(scale=2):
                back_btn = gr.Button("⬅️ Analyze Another Song", variant="secondary")
            gr.Column(scale=1)

    # Logic & Events
    def check_inputs(audio, lyrics, year):
        """
        Validates the UI inputs to determine if the analysis button should be enabled.

        Args:
            audio: The uploaded audio file object.
            lyrics (str): The provided lyrics text.
            year: The release year numerical input.

        Returns:
            gr.update: A Gradio update object altering the interactivity of the analyze button.
        """
        if audio is not None and lyrics and lyrics.strip() != "" and year is not None:
            return gr.update(interactive=True)
        return gr.update(interactive=False)

    for inp in [audio_input, lyrics_input, year_input]:
        inp.change(fn=check_inputs, inputs=[audio_input, lyrics_input, year_input], outputs=analyze_btn)

    analyze_btn.click(
        fn=process_song_workflow,
        inputs=[audio_input, lyrics_input, year_input],
        outputs=[main_screen, loading_screen, results_screen, error_box, scores_display, recommendations_display, spider_plot]
    )

    def go_back():
        """
        Resets the application interface back to its initial state, clearing inputs
        and returning the user to the main screen for a new analysis.

        Returns:
            tuple: Multiple Gradio update objects that clear input fields and toggle component visibility.
        """
        return (
            None, "", 2024,
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(interactive=False)
        )

    back_btn.click(
        fn=go_back,
        inputs=[],
        outputs=[audio_input, lyrics_input, year_input, main_screen, results_screen, analyze_btn]
    )

print("Launching PopScore Web Interface...")
demo.launch(share=True, debug=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


☁️ Downloading model weights from Hugging Face Hub...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


best_hit_predictor_sharp.pth:   0%|          | 0.00/101M [00:00<?, ?B/s]

best_surrogate_model.pth:   0%|          | 0.00/41.2k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/websockets/legacy/__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/websockets/websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


* Running on public URL: https://58fb1de97dc4a32104.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1341: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1341: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprec